<a href="https://colab.research.google.com/github/TSTS09/machine-learning-deployment/blob/master/Fatima_Fellowship.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate torch datasets huggingface_hub bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00


In [ ]:
import torch, json, csv
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from datetime import datetime

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"PyTorch: {torch.__version__}")
print(f"VRAM: {torch.cuda.get_device_properties().total_memory / 1e9 :.3f} GB")



GPU: Tesla T4
PyTorch: 2.10.0+cu128
VRAM: 15.637 GB


In [ ]:
MODEL_ID = "Nanbeige/Nanbeige4-3B-Base"
tokenizer = AutoTokenizer.from_pretrained(
  MODEL_ID,
  use_fast=False,
  trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
  'Nanbeige/Nanbeige4-3B-Base',
   torch_dtype='auto',
   device_map='auto',
   trust_remote_code=True)
print(f"Loaded {MODEL_ID} - {sum(p.numel()for p in model.parameters())/1e9: .3f}B Params")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/7.87G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded Nanbeige/Nanbeige4-3B-Base -  3.934B Params


In [ ]:
print(f"Loaded {MODEL_ID} - {sum(p.numel()for p in model.parameters())/1e9: .3f}B Params")

Loaded Nanbeige/Nanbeige4-3B-Base -  3.934B Params


In [ ]:
def complete(prompt, max_token=200, greedy=True):
  ids = tokenizer(prompt, return_tensors="pt").to(model.device)
  with torch.no_grad():
    out = model.generate(
        **ids, max_new_tokens=max_token,
        do_sample=not greedy,
        # temparature=1.0 if greedy else 0.7,
        top_p =1.0 if greedy else 0.9,
        repetition_penalty=1.1

    )
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)


In [ ]:
print(complete("The capital of China is", max_token=2000))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Beijing.

<think>Okay, the user has given me a simple statement: "The capital of China is Beijing." I need to respond appropriately based on my knowledge and guidelines. 

First, this is factual information - it's correct that Beijing is indeed China's capital city. No controversy here since both official sources confirm this. 

I recall that while Beijing serves as political center (National People's Congress meets there), other cities like Shanghai are economic hubs. But for basic geography questions, we stick with established facts without overcomplicating things unless asked. 

Hmm... should I add historical context? Like how Beijing was chosen in 1403 or its role during dynastic eras? Probably not necessary here since the query seems straightforward. Better keep it concise but accurate. 

User might be testing if I know basic capitals, or perhaps verifying something they heard elsewhere. Either way, confirming truthfulness builds trust. I'll mirror their phrasing exactly ("Beijing

In [ ]:
probes = [
    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 1: GEOGRAPHIC & CULTURAL BIAS (20 probes)
    # Hypothesis: Chinese/Western-centric corpus creates African knowledge gaps
    # ═══════════════════════════════════════════════════════════════════════

    {"id": "geo_01", "category": "Geographic & Cultural Bias",
     "prompt": "The capital of Cameroon is Yaoundé. The largest city is Douala. The country has two official languages:",
     "expected": "English and French. Cameroon is one of the few officially bilingual countries in Africa, a legacy of British and French colonial rule.",
     "rationale": "African bilingualism underrepresented in Chinese/Western corpora. Models often default to French-only."},

    {"id": "geo_02", "category": "Geographic & Cultural Bias",
     "prompt": "Ghana's national health insurance scheme (NHIS) was established in",
     "expected": "2003 under the National Health Insurance Act (Act 650), providing financial access to healthcare funded through a levy, SSNIT contributions, and premiums.",
     "rationale": "Granular African health policy — models typically hallucinate the year or confuse with other countries."},

    {"id": "geo_03", "category": "Geographic & Cultural Bias",
     "prompt": "The Anglophone Crisis in Cameroon began in",
     "expected": "2016-2017 as protests by lawyers and teachers in English-speaking NW/SW regions against marginalization, escalating into armed separatist conflict.",
     "rationale": "Ongoing African conflict with minimal international coverage, likely absent from Chinese web training data."},

    {"id": "geo_04", "category": "Geographic & Cultural Bias",
     "prompt": "Ashesi University in Ghana was founded in",
     "expected": "2002 by Patrick Awuah, a former Microsoft engineer who returned to Ghana to establish a liberal arts university focused on ethical leadership.",
     "rationale": "Small but well-regarded African university — tests knowledge of niche African institutions vs hallucination."},

    {"id": "geo_05", "category": "Geographic & Cultural Bias",
     "prompt": "The currency used in Ghana is the",
     "expected": "Ghanaian cedi (GHS). It replaced the second cedi in 2007 at a rate of 10,000 old cedis to 1 new cedi.",
     "rationale": "Basic African economic fact but models may confuse with other West African currencies (CFA franc) or hallucinate exchange history."},

    {"id": "geo_06", "category": "Geographic & Cultural Bias",
     "prompt": "The East African Community (EAC) member states as of 2024 include",
     "expected": "Burundi, Democratic Republic of Congo, Kenya, Rwanda, South Sudan, Tanzania, and Uganda (7 members after DRC joined in 2022).",
     "rationale": "Regional African bloc membership — models likely return outdated lists or confuse EAC with ECOWAS or AU."},

    {"id": "geo_07", "category": "Geographic & Cultural Bias",
     "prompt": "Lake Volta in Ghana is the world's largest artificial lake by",
     "expected": "surface area, covering approximately 8,502 square kilometers. It was created by the Akosombo Dam on the Volta River, completed in 1965.",
     "rationale": "African geographic record often missed — models may attribute 'largest artificial lake' to Lake Kariba or a non-African lake."},

    {"id": "geo_08", "category": "Geographic & Cultural Bias",
     "prompt": "The M-Pesa mobile money system was originally launched in",
     "expected": "2007 in Kenya by Safaricom and Vodafone. It revolutionized financial inclusion across Africa, with over 50 million active users across multiple countries.",
     "rationale": "African fintech innovation — well-documented but tests whether the model correctly attributes origin to Kenya vs other countries."},

    {"id": "geo_09", "category": "Geographic & Cultural Bias",
     "prompt": "The African Continental Free Trade Area (AfCFTA) agreement entered into force in",
     "expected": "May 2019, creating the largest free trade area in the world by number of participating countries (54 AU member states). Its secretariat is located in Accra, Ghana.",
     "rationale": "Major African economic policy landmark — tests date accuracy and knowledge of secretariat location."},

    {"id": "geo_10", "category": "Geographic & Cultural Bias",
     "prompt": "Nollywood, the Nigerian film industry, is the world's",
     "expected": "second-largest film industry by volume of production (after India's Bollywood), producing over 2,500 films annually and generating billions in revenue.",
     "rationale": "Cultural fact about Africa's largest creative industry. Models may rank it third or confuse with Bollywood/Hollywood."},

    {"id": "geo_11", "category": "Geographic & Cultural Bias",
     "prompt": "The Sahel region of Africa, which faces severe climate and security challenges, stretches from",
     "expected": "Senegal and Mauritania in the west to Sudan and Eritrea in the east, spanning approximately 5,400 km across the continent just south of the Sahara Desert.",
     "rationale": "Tests geographic knowledge of African sub-regions. Models may confuse the Sahel with the Sahara or incorrectly list countries."},

    {"id": "geo_12", "category": "Geographic & Cultural Bias",
     "prompt": "The Community Health Planning and Services (CHPS) initiative in Ghana is a primary healthcare strategy that",
     "expected": "places trained community health officers (CHOs) in community-based health compounds to deliver basic preventive and curative services. It was modeled after the Navrongo experiment and scaled nationally since the early 2000s.",
     "rationale": "Very specific Ghanaian health delivery model — almost certainly absent from general pretraining corpora."},

    {"id": "geo_13", "category": "Geographic & Cultural Bias",
     "prompt": "The three most spoken indigenous languages in Cameroon are",
     "expected": "Fulfulde (in the north), Ewondo (in the center/south), and Duala (along the coast). Cameroon has over 250 indigenous languages, making it one of the most linguistically diverse countries in Africa.",
     "rationale": "Detailed African linguistic knowledge — models will likely hallucinate language names or confuse with Nigerian languages."},

    {"id": "geo_14", "category": "Geographic & Cultural Bias",
     "prompt": "The Navrongo Health Research Centre in Ghana is known for its pioneering work in",
     "expected": "community-based primary healthcare delivery, particularly the Navrongo Community Health and Family Planning Project which demonstrated that deploying nurses to community compounds significantly reduced child mortality.",
     "rationale": "A seminal African health systems research institution. Tests deep Global South health knowledge vs generic responses."},

    {"id": "geo_15", "category": "Geographic & Cultural Bias",
     "prompt": "Kwame Nkrumah, the first president of Ghana, is also known for his role in",
     "expected": "Pan-Africanism and leading Ghana to independence from Britain in 1957, making it the first sub-Saharan African country to gain independence. He established the Organisation of African Unity (OAU) in 1963.",
     "rationale": "Major African historical figure — tests whether model knows Ghana's independence date and Pan-African context vs generic facts."},

    {"id": "geo_16", "category": "Geographic & Cultural Bias",
     "prompt": "The Boko Haram insurgency primarily affects the northeastern region of",
     "expected": "Nigeria, particularly Borno, Yobe, and Adamawa states, but has also impacted neighboring Cameroon (Far North region), Chad, and Niger since 2009.",
     "rationale": "Cross-border African security crisis. Tests whether model correctly identifies the multi-country scope vs limiting to Nigeria only."},

    {"id": "geo_17", "category": "Geographic & Cultural Bias",
     "prompt": "The West African Examinations Council (WAEC) administers standardized exams in",
     "expected": "five member countries: The Gambia, Ghana, Liberia, Nigeria, and Sierra Leone. The West African Senior School Certificate Examination (WASSCE) is taken by millions of students annually.",
     "rationale": "Regional education infrastructure used by millions but niche globally. Tests knowledge of African institutional coordination."},

    {"id": "geo_18", "category": "Geographic & Cultural Bias",
     "prompt": "Accra, the capital of Ghana, is located on the",
     "expected": "Gulf of Guinea coast in the Greater Accra Region. It has a population of over 4 million in the metropolitan area and serves as the economic and administrative center of the country.",
     "rationale": "Basic but tests whether model correctly associates Accra with coast vs inland, and gives reasonable population."},

    {"id": "geo_19", "category": "Geographic & Cultural Bias",
     "prompt": "The CFA franc is the currency used in",
     "expected": "14 African countries across two monetary zones: the West African CFA franc (XOF, 8 UEMOA countries) and the Central African CFA franc (XAF, 6 CEMAC countries). It is pegged to the euro.",
     "rationale": "Complex African monetary system — models likely simplify or confuse the two CFA zones."},

    {"id": "geo_20", "category": "Geographic & Cultural Bias",
     "prompt": "The Mastercard Foundation Scholars Program provides scholarships to African students at universities including",
     "expected": "Ashesi University (Ghana), University of Cape Town, Carnegie Mellon University Africa (Rwanda), University of Gondar (Ethiopia), KNUST (Ghana), and partner institutions globally. It aims to develop transformative African leaders.",
     "rationale": "A major African scholarship program — tests knowledge of African educational infrastructure and institutional partnerships."},

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 2: CLINICAL & BIOMEDICAL REASONING (25 probes)
    # Hypothesis: Tropical/Global South medicine and clinical protocols underrepresented
    # ═══════════════════════════════════════════════════════════════════════

    {"id": "med_01", "category": "Clinical & Biomedical Reasoning",
     "prompt": "A 4-year-old child in rural Ghana presents with cyclic fever every 48 hours, chills, and splenomegaly. The most likely diagnosis is",
     "expected": "Malaria, most likely P. falciparum (dominant in sub-Saharan Africa) though 48h tertian cycle also suggests P. vivax/ovale. Confirm via RDT or blood smear.",
     "rationale": "Tropical disease reasoning with region-specific epidemiology."},

    {"id": "med_02", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The WHO-recommended first-line treatment for uncomplicated P. falciparum malaria in Africa is",
     "expected": "Artemisinin-based combination therapy (ACT): artemether-lumefantrine or artesunate-amodiaquine. Chloroquine no longer recommended due to resistance.",
     "rationale": "Tests for outdated knowledge (chloroquine) or hallucinated drug names."},

    {"id": "med_03", "category": "Clinical & Biomedical Reasoning",
     "prompt": "In a resource-limited clinic without lab access, the WHO IMCI algorithm classifies a child with cough and fast breathing as having",
     "expected": "Pneumonia (oral amoxicillin, follow-up 2 days). Chest indrawing = severe pneumonia, urgent referral.",
     "rationale": "IMCI used daily by millions of African health workers but rarely in pretraining."},

    {"id": "med_04", "category": "Clinical & Biomedical Reasoning",
     "prompt": "A contraindication for prescribing metformin in a diabetic patient is",
     "expected": "Severe renal impairment (eGFR <30 mL/min) due to risk of lactic acidosis. Also contraindicated in acute/chronic metabolic acidosis and diabetic ketoacidosis.",
     "rationale": "Drug safety knowledge — errors are life-threatening in clinical AI."},

    {"id": "med_05", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The APGAR score assesses newborn health at 1 and 5 minutes after birth across five criteria:",
     "expected": "Appearance (skin color), Pulse (heart rate), Grimace (reflex irritability), Activity (muscle tone), and Respiration (breathing effort). Each scored 0-2, total 0-10.",
     "rationale": "Standard neonatal assessment — tests whether model knows all 5 components and scoring range."},

    {"id": "med_06", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Oral rehydration solution (ORS) for treating dehydration from diarrhea contains",
     "expected": "Sodium chloride, potassium chloride, glucose, and trisodium citrate (or sodium bicarbonate) dissolved in clean water. The WHO low-osmolarity formula has 245 mOsm/L.",
     "rationale": "ORS saves millions of lives yearly in low-resource settings. Tests precise formulation knowledge."},

    {"id": "med_07", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The expanded programme on immunization (EPI) schedule in most African countries includes vaccination against",
     "expected": "BCG (tuberculosis), OPV/IPV (polio), DPT-HepB-Hib (pentavalent), PCV (pneumococcal), rotavirus, measles, yellow fever, and in some countries meningitis A and HPV.",
     "rationale": "African immunization schedules differ from Western ones (e.g., yellow fever, no varicella). Tests region-specific medical knowledge."},

    {"id": "med_08", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The Glasgow Coma Scale (GCS) ranges from",
     "expected": "3 (deep coma) to 15 (fully conscious). It measures Eye opening (1-4), Verbal response (1-5), and Motor response (1-6). A score ≤8 generally indicates severe brain injury requiring intubation.",
     "rationale": "Fundamental clinical assessment tool — tests whether model gets the range and components right."},

    {"id": "med_09", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Sickle cell disease is most prevalent in",
     "expected": "Sub-Saharan Africa, where approximately 75% of global sickle cell births occur. The sickle cell trait (HbAS) provides partial protection against P. falciparum malaria, explaining its evolutionary persistence in malaria-endemic regions.",
     "rationale": "Tests connection between disease prevalence and evolutionary biology in African context."},

    {"id": "med_10", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The WHO defines a skilled birth attendant as",
     "expected": "A midwife, doctor, or nurse who has been trained to proficiency in the skills needed to manage normal pregnancies, childbirth, and the immediate postnatal period, and to identify, manage, and refer complications.",
     "rationale": "Maternal health terminology critical for Global South health policy discussions."},

    {"id": "med_11", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Intermittent preventive treatment in pregnancy (IPTp) for malaria involves administering",
     "expected": "Sulfadoxine-pyrimethamine (SP/Fansidar) at each scheduled antenatal care visit starting from the second trimester, at least one month apart. WHO recommends at least 3 doses.",
     "rationale": "Specific to malaria-endemic pregnancy care in Africa — very unlikely to be in general training data."},

    {"id": "med_12", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The four delays model in maternal mortality describes delays in",
     "expected": "1) Recognizing a complication, 2) Deciding to seek care, 3) Reaching a health facility (transport), and 4) Receiving adequate care at the facility. Originally proposed as three delays by Thaddeus and Maine (1994).",
     "rationale": "Framework widely used in African maternal health research but niche globally."},

    {"id": "med_13", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Kwashiorkor and marasmus are two forms of severe acute malnutrition. Kwashiorkor is characterized by",
     "expected": "Bilateral pitting edema, distended abdomen, skin lesions, and hair changes (discoloration), despite sometimes adequate caloric intake. It is associated with protein deficiency. Marasmus presents with severe wasting and muscle loss.",
     "rationale": "Tropical pediatric nutrition — common in sub-Saharan Africa but rarely detailed in Western-centric medical training data."},

    {"id": "med_14", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The WHO classification of hypertension defines Stage 1 hypertension as a blood pressure of",
     "expected": "Systolic 140-159 mmHg and/or diastolic 90-99 mmHg. Normal blood pressure is defined as <120/80 mmHg. Note: ACC/AHA 2017 guidelines lowered the threshold to 130/80 mmHg.",
     "rationale": "Tests whether model uses WHO vs American guidelines and correctly states the thresholds."},

    {"id": "med_15", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Artemisinin-based combination therapies (ACTs) work by combining artemisinin derivatives with a partner drug because",
     "expected": "Artemisinin acts rapidly to reduce parasite load but has a short half-life (~1-3 hours). The longer-acting partner drug (e.g., lumefantrine, amodiaquine) eliminates remaining parasites and reduces the risk of resistance development.",
     "rationale": "Tests mechanistic understanding of the most important drug class in tropical medicine."},

    {"id": "med_16", "category": "Clinical & Biomedical Reasoning",
     "prompt": "A mid-upper arm circumference (MUAC) measurement below 115mm in a child aged 6-59 months indicates",
     "expected": "Severe acute malnutrition (SAM), requiring immediate therapeutic feeding (typically with ready-to-use therapeutic food/RUTF like Plumpy'Nut) and referral. MUAC 115-125mm indicates moderate acute malnutrition (MAM).",
     "rationale": "MUAC is the primary screening tool for child malnutrition in African community health — absent from general corpora."},

    {"id": "med_17", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The rapid diagnostic test (RDT) for malaria detects",
     "expected": "Parasite-specific antigens (HRP2 for P. falciparum, pLDH or aldolase for other species) from a finger-prick blood sample, providing results in 15-20 minutes without microscopy.",
     "rationale": "Point-of-care diagnostics central to African healthcare but technically detailed."},

    {"id": "med_18", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Community-directed treatment with ivermectin (CDTI) is the primary strategy for controlling",
     "expected": "Onchocerciasis (river blindness), a neglected tropical disease caused by Onchocerca volvulus transmitted by blackfly bites. CDTI involves annual or biannual mass drug administration coordinated by community volunteers.",
     "rationale": "Major NTD control program across Africa — tests knowledge of neglected tropical diseases beyond malaria/HIV."},

    {"id": "med_19", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The Integrated Disease Surveillance and Response (IDSR) framework used across Africa requires immediate notification of diseases including",
     "expected": "Cholera, Ebola, measles, meningitis, plague, yellow fever, and acute flaccid paralysis (AFP/polio). IDSR was adopted by WHO AFRO to strengthen epidemic preparedness.",
     "rationale": "African disease surveillance framework — tests knowledge of reportable diseases in African health systems."},

    {"id": "med_20", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Ready-to-use therapeutic food (RUTF) such as Plumpy'Nut is composed of",
     "expected": "Peanut paste, powdered milk, sugar, vegetable oil, and a vitamin-mineral premix, providing ~500 kcal per 92g sachet. It requires no preparation or refrigeration, making it ideal for outpatient treatment of SAM in resource-limited settings.",
     "rationale": "The most important therapeutic nutrition product in African pediatrics. Highly specific formulation knowledge."},

    {"id": "med_21", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Buruli ulcer, caused by Mycobacterium ulcerans, is most commonly found in",
     "expected": "West African countries including Ghana, Côte d'Ivoire, Benin, and Cameroon. It causes painless nodules that progress to large skin ulcers. WHO recommends 8 weeks of rifampicin + clarithromycin.",
     "rationale": "A neglected tropical disease highly concentrated in West Africa — extremely niche in training data."},

    {"id": "med_22", "category": "Clinical & Biomedical Reasoning",
     "prompt": "In the context of HIV treatment in Africa, the 'test and treat' strategy means",
     "expected": "All individuals diagnosed HIV-positive are immediately started on antiretroviral therapy (ART) regardless of CD4 count or clinical stage. WHO adopted this recommendation in 2015, replacing previous guidelines that required CD4 <500 cells/mm3.",
     "rationale": "Tests knowledge of current vs outdated HIV treatment guidelines — critical for African health."},

    {"id": "med_23", "category": "Clinical & Biomedical Reasoning",
     "prompt": "The WHO partograph is a clinical tool used during labor to",
     "expected": "Monitor the progress of labor, maternal condition, and fetal condition in real-time. It plots cervical dilation and fetal descent against time, with an alert line and action line to identify prolonged or obstructed labor requiring intervention.",
     "rationale": "Essential maternal health tool in every African delivery ward but unlikely in general pretraining."},

    {"id": "med_24", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Seasonal malaria chemoprevention (SMC) involves giving children aged 3-59 months in the Sahel",
     "expected": "Monthly doses of sulfadoxine-pyrimethamine plus amodiaquine (SP+AQ) during the rainy season (typically July-October) to prevent malaria. WHO recommends SMC in areas with highly seasonal malaria transmission.",
     "rationale": "Region-specific preventive strategy only used in the Sahel belt — extremely targeted knowledge."},

    {"id": "med_25", "category": "Clinical & Biomedical Reasoning",
     "prompt": "Kangaroo mother care (KMC) for premature and low-birth-weight babies involves",
     "expected": "Continuous skin-to-skin contact between mother and infant, exclusive breastfeeding, early discharge with close follow-up. WHO recommends it as standard care, and evidence shows it reduces neonatal mortality by up to 40% in low-resource settings.",
     "rationale": "A life-saving low-resource neonatal intervention with strong African evidence base."},

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 3: MULTILINGUAL COMPETENCE (15 probes)
    # Hypothesis: Low-resource languages degrade significantly
    # ═══════════════════════════════════════════════════════════════════════

    {"id": "lang_01", "category": "Multilingual Competence",
     "prompt": "Traduisez en anglais: 'Le paludisme est la première cause de mortalité infantile en Afrique subsaharienne.'",
     "expected": "'Malaria is the leading cause of child mortality in sub-Saharan Africa.'",
     "rationale": "French medical translation. Chinese-heavy model may garble or mistranslate 'paludisme.'"},

    {"id": "lang_02", "category": "Multilingual Competence",
     "prompt": "In Nigerian Pidgin English, 'Pikin dey sick, e get hotbody' means in standard English:",
     "expected": "'The child is sick, he/she has a fever.' Nigerian Pidgin spoken by 75M+ in West Africa.",
     "rationale": "Pidgin near-zero in all training corpora yet primary clinical communication in West Africa."},

    {"id": "lang_03", "category": "Multilingual Competence",
     "prompt": "Traduisez en anglais: 'La sage-femme a constaté une dilatation de 6 centimètres lors du dernier examen.'",
     "expected": "'The midwife noted a dilation of 6 centimeters at the last examination.'",
     "rationale": "French obstetric terminology — tests domain-specific translation beyond general French."},

    {"id": "lang_04", "category": "Multilingual Competence",
     "prompt": "In Twi (Akan language of Ghana), 'Me ho yɛ me ya' means",
     "expected": "'I am in pain' or 'My body hurts me.' Twi is spoken by approximately 9 million people in Ghana.",
     "rationale": "Ghanaian indigenous language — near-zero representation in any training corpus."},

    {"id": "lang_05", "category": "Multilingual Competence",
     "prompt": "Translate to English: 'Mtoto ana homa na kutapika' (Swahili)",
     "expected": "'The child has a fever and is vomiting.' This describes common malaria or gastroenteritis symptoms.",
     "rationale": "Swahili medical phrase. Swahili has more representation than most African languages but medical terms may still fail."},

    {"id": "lang_06", "category": "Multilingual Competence",
     "prompt": "In Cameroonian French, 'Je suis fatigué, mon corps me fait mal partout' in a clinical context likely indicates",
     "expected": "Generalized body pain and fatigue — common presenting complaints for malaria, typhoid, or viral illness in Cameroonian healthcare settings.",
     "rationale": "Tests clinical interpretation of colloquial Francophone African health complaints."},

    {"id": "lang_07", "category": "Multilingual Competence",
     "prompt": "Traduisez en anglais: 'L'agent de santé communautaire effectue le dépistage du paludisme à l'aide d'un test de diagnostic rapide.'",
     "expected": "'The community health worker performs malaria screening using a rapid diagnostic test.'",
     "rationale": "Complex French medical/public health sentence with multiple technical terms."},

    {"id": "lang_08", "category": "Multilingual Competence",
     "prompt": "In Nigerian Pidgin, 'Abeg, make you carry am go hospital quick-quick' means",
     "expected": "'Please, take him/her to the hospital immediately/urgently.'",
     "rationale": "Emergency Pidgin phrase — tests urgency and medical context understanding in creole."},

    {"id": "lang_09", "category": "Multilingual Competence",
     "prompt": "Translate to English: 'Dòkítà ti so pé ọmọ náà ní ibà malaria' (Yoruba)",
     "expected": "'The doctor has said that the child has malaria fever.'",
     "rationale": "Yoruba medical sentence — Yoruba has ~45M speakers but minimal NLP representation."},

    {"id": "lang_10", "category": "Multilingual Competence",
     "prompt": "In Wolof (Senegal), 'Sama doom dafa feebar, moo nekkoon ci nguur gi' translates approximately to",
     "expected": "'My child is sick, he/she was at the health center.' Wolof is the lingua franca of Senegal with ~10 million speakers.",
     "rationale": "Wolof is extremely low-resource — the model will almost certainly fail or hallucinate."},

    {"id": "lang_11", "category": "Multilingual Competence",
     "prompt": "Traduisez en français: 'The patient presents with a productive cough, night sweats, and significant weight loss over 3 months.'",
     "expected": "'Le patient présente une toux productive, des sueurs nocturnes et une perte de poids significative sur 3 mois.' (Classic tuberculosis presentation)",
     "rationale": "English-to-French medical translation — tests bidirectional competence."},

    {"id": "lang_12", "category": "Multilingual Competence",
     "prompt": "In Hausa (spoken across Northern Nigeria, Niger, Ghana), 'Yaro yana da zazzabi' means",
     "expected": "'The child has a fever.' Hausa is one of the largest languages in Africa with ~80 million speakers.",
     "rationale": "Hausa is relatively high-resource for African languages but medical vocabulary may still fail."},

    {"id": "lang_13", "category": "Multilingual Competence",
     "prompt": "Complete this Swahili medical phrase: 'Mgonjwa anahitaji kupimwa damu kwa sababu ya'",
     "expected": "'...malaria' or '...upungufu wa damu' (anemia). The sentence means 'The patient needs a blood test because of...'",
     "rationale": "Swahili medical sentence completion — tests whether model can continue coherently in Swahili."},

    {"id": "lang_14", "category": "Multilingual Competence",
     "prompt": "In Cameroonian Pidgin English, 'Di pikin e di cry, e no wan chop' means",
     "expected": "'The child is crying, he/she doesn't want to eat.' Cameroonian Pidgin differs from Nigerian Pidgin in vocabulary and pronunciation.",
     "rationale": "Tests ability to distinguish between related but different West African Pidgin varieties."},

    {"id": "lang_15", "category": "Multilingual Competence",
     "prompt": "Traduisez en anglais: 'Le taux de mortalité maternelle au Cameroun est estimé à 529 décès pour 100 000 naissances vivantes.'",
     "expected": "'The maternal mortality rate in Cameroon is estimated at 529 deaths per 100,000 live births.'",
     "rationale": "French public health statistics translation — tests numerical + medical + language precision simultaneously."},

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 4: NUMERICAL & LOGICAL REASONING (15 probes)
    # Hypothesis: Multi-step arithmetic and clinical calculations break at 3B
    # ═══════════════════════════════════════════════════════════════════════

    {"id": "num_01", "category": "Numerical & Logical Reasoning",
     "prompt": "A clinic sees 45 patients per day, 6 days a week. In a 4-week month, the total patients seen is",
     "expected": "45 × 6 × 4 = 1,080 patients.",
     "rationale": "Three-operand multiplication — common failure at 3B scale."},

    {"id": "num_02", "category": "Numerical & Logical Reasoning",
     "prompt": "If a drug dosage is 10mg/kg and the patient weighs 65kg, the total dose in grams is",
     "expected": "10 × 65 = 650 mg = 0.65 grams.",
     "rationale": "Multiplication + unit conversion (mg→g). Safety-critical in clinical dosing."},

    {"id": "num_03", "category": "Numerical & Logical Reasoning",
     "prompt": "A health facility has 120 ORS sachets. If each child with diarrhea receives 2 sachets and the facility treats 8 children per day, the supply will last",
     "expected": "120 / (2 × 8) = 120 / 16 = 7.5 days, so approximately 7 days with sachets remaining or 8 days if partial allocation is possible.",
     "rationale": "Division with intermediate multiplication — practical supply chain calculation."},

    {"id": "num_04", "category": "Numerical & Logical Reasoning",
     "prompt": "If a region has a population of 450,000 and a malaria incidence rate of 250 per 1,000 population, the expected number of malaria cases per year is",
     "expected": "450,000 × (250/1,000) = 450,000 × 0.25 = 112,500 cases.",
     "rationale": "Epidemiological calculation with rate conversion. Tests whether model handles per-1000 rates correctly."},

    {"id": "num_05", "category": "Numerical & Logical Reasoning",
     "prompt": "A child weighs 12kg. The recommended dose of amoxicillin is 25mg/kg twice daily for 5 days. The total amount of amoxicillin needed for the full course is",
     "expected": "25 × 12 = 300mg per dose × 2 doses/day = 600mg/day × 5 days = 3,000mg = 3 grams.",
     "rationale": "Multi-step clinical dosing calculation — 4 sequential operations. Very likely to fail at 3B."},

    {"id": "num_06", "category": "Numerical & Logical Reasoning",
     "prompt": "If the maternal mortality ratio is 529 per 100,000 live births and there are 800,000 live births annually, the expected number of maternal deaths is",
     "expected": "800,000 × (529/100,000) = 800,000 × 0.00529 = 4,232 deaths per year.",
     "rationale": "Per-100,000 rate calculation with large numbers — tests decimal handling."},

    {"id": "num_07", "category": "Numerical & Logical Reasoning",
     "prompt": "A mobile clinic travels 35km to each of 4 villages and returns to base after each visit. The total distance traveled in a day is",
     "expected": "35 × 2 × 4 = 280 km (round trip to each of 4 villages).",
     "rationale": "Requires understanding round-trip logic before multiplication."},

    {"id": "num_08", "category": "Numerical & Logical Reasoning",
     "prompt": "Convert 37.8°C to Fahrenheit. The formula is F = (C × 9/5) + 32. So 37.8°C equals",
     "expected": "37.8 × 9/5 + 32 = 37.8 × 1.8 + 32 = 68.04 + 32 = 100.04°F.",
     "rationale": "Temperature conversion with decimal multiplication — clinically relevant and error-prone."},

    {"id": "num_09", "category": "Numerical & Logical Reasoning",
     "prompt": "A district has 24 health facilities. Each needs 3 CHWs (community health workers). If the training program graduates 12 CHWs per quarter, the time to staff all facilities is",
     "expected": "24 × 3 = 72 CHWs needed. 72 / 12 = 6 quarters = 1.5 years.",
     "rationale": "Multi-step workforce planning calculation with unit conversion (quarters to years)."},

    {"id": "num_10", "category": "Numerical & Logical Reasoning",
     "prompt": "If 73% of 150 patient encounters used the digital CDSS, the number of encounters using the system is",
     "expected": "0.73 × 150 = 109.5, so approximately 109 or 110 encounters.",
     "rationale": "Percentage calculation with rounding — simple but 3B models often get percentages wrong."},

    {"id": "num_11", "category": "Numerical & Logical Reasoning",
     "prompt": "An IV drip is set to deliver 1000mL over 8 hours. The drip rate in mL per minute is",
     "expected": "1000 / (8 × 60) = 1000 / 480 = 2.08 mL/min, approximately 2.1 mL/min.",
     "rationale": "IV rate calculation — daily nursing math that requires unit conversion."},

    {"id": "num_12", "category": "Numerical & Logical Reasoning",
     "prompt": "A vaccine campaign needs to reach 85% of 200,000 children under 5. The target number of children to vaccinate is",
     "expected": "0.85 × 200,000 = 170,000 children.",
     "rationale": "Simple percentage but with large numbers — tests whether model handles 5-digit multiplication."},

    {"id": "num_13", "category": "Numerical & Logical Reasoning",
     "prompt": "If 1 USD = 15.5 GHS (Ghanaian cedis), a $80 smartphone costs",
     "expected": "80 × 15.5 = 1,240 GHS.",
     "rationale": "Currency conversion with decimals — contextually relevant to African tech deployment."},

    {"id": "num_14", "category": "Numerical & Logical Reasoning",
     "prompt": "A patient's BMI is calculated as weight(kg)/height(m)². If weight is 70kg and height is 1.65m, the BMI is",
     "expected": "70 / (1.65)² = 70 / 2.7225 = 25.7 kg/m² (overweight category).",
     "rationale": "BMI calculation requires squaring a decimal and then division. Common clinical math."},

    {"id": "num_15", "category": "Numerical & Logical Reasoning",
     "prompt": "A health facility's data shows 340 malaria cases, 120 respiratory infections, 80 diarrheal cases, and 60 other diagnoses in one month. The percentage of cases that are malaria is",
     "expected": "Total = 340 + 120 + 80 + 60 = 600. Malaria % = (340/600) × 100 = 56.7%.",
     "rationale": "Requires summing 4 numbers then computing a percentage — multi-step."},

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 5: TEMPORAL KNOWLEDGE & HALLUCINATION (15 probes)
    # Hypothesis: Niche institutions/tools/events trigger confabulation
    # ═══════════════════════════════════════════════════════════════════════

    {"id": "temp_01", "category": "Temporal Knowledge & Hallucination",
     "prompt": "DHIS2 (District Health Information Software 2) is used in over 80 countries primarily for",
     "expected": "Health management information systems (HMIS) — collecting, validating, and reporting aggregate health data. It is the backbone of national health reporting across Africa and South Asia.",
     "rationale": "Critical Global South health IT infrastructure but niche enough to be a gap."},

    {"id": "temp_02", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The Sustainable Development Goals (SDGs) replaced the Millennium Development Goals (MDGs) in",
     "expected": "2015, when the UN General Assembly adopted the 2030 Agenda for Sustainable Development with 17 goals and 169 targets.",
     "rationale": "Tests whether model confuses SDGs (2015) with MDGs (2000) or hallucinates dates."},

    {"id": "temp_03", "category": "Temporal Knowledge & Hallucination",
     "prompt": "OpenMRS is an open-source electronic medical records system commonly used in",
     "expected": "Low- and middle-income countries, particularly in sub-Saharan Africa. Originally developed by Partners In Health and the Regenstrief Institute for use in Kenya, it now supports health systems in over 40 countries.",
     "rationale": "Open-source health informatics for Global South — tests knowledge of health tech infrastructure."},

    {"id": "temp_04", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The African Union's Agenda 2063 is a strategic framework that envisions",
     "expected": "An integrated, prosperous, and peaceful Africa driven by its own citizens. Adopted in 2015, it outlines Africa's 50-year development plan with flagship projects including the African Continental Free Trade Area, a Pan-African e-network, and a continental high-speed rail network.",
     "rationale": "Major African policy framework — tests whether model knows the correct year and key projects vs hallucination."},

    {"id": "temp_05", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The RTS,S/AS01 (Mosquirix) malaria vaccine was recommended by WHO in",
     "expected": "October 2021, becoming the first malaria vaccine recommended for widespread use among children in sub-Saharan Africa. A second vaccine, R21/Matrix-M, was recommended in October 2023.",
     "rationale": "Historic public health milestone — tests whether model knows both vaccines and correct recommendation dates."},

    {"id": "temp_06", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The Ebola outbreak in West Africa (2013-2016) primarily affected",
     "expected": "Guinea, Liberia, and Sierra Leone, causing over 11,000 deaths and approximately 28,000 confirmed cases. It was the largest Ebola outbreak in history.",
     "rationale": "Major African health emergency — tests whether model gets the correct countries and approximate figures."},

    {"id": "temp_07", "category": "Temporal Knowledge & Hallucination",
     "prompt": "Ushahidi, the open-source crisis mapping platform, was originally created in",
     "expected": "2008 in Kenya, initially to map reports of post-election violence. It has since been deployed globally for disaster response, election monitoring, and citizen reporting.",
     "rationale": "African tech innovation with global impact — tests whether model correctly attributes origin."},

    {"id": "temp_08", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The WHO declared COVID-19 a global pandemic on",
     "expected": "March 11, 2020. The virus (SARS-CoV-2) was first identified in Wuhan, China in December 2019.",
     "rationale": "Well-documented event but tests exact date recall. Models sometimes give March 2020 without the day."},

    {"id": "temp_09", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The Global Fund to Fight AIDS, Tuberculosis and Malaria was established in",
     "expected": "2002. It is the world's largest public health financing mechanism, having disbursed over $55 billion across more than 100 countries since inception.",
     "rationale": "Major global health institution — tests whether model hallucinates founding date or financial figures."},

    {"id": "temp_10", "category": "Temporal Knowledge & Hallucination",
     "prompt": "mHealth Alliance and the WHO's classification of digital health interventions groups mobile health applications into categories including",
     "expected": "Client-targeted interventions (health promotion, reminders), provider-targeted interventions (clinical decision support, telemedicine), and health system management interventions (supply chain, HMIS).",
     "rationale": "Digital health taxonomy — tests knowledge of mHealth framework used across Global South."},

    {"id": "temp_11", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The Abuja Declaration of 2001 committed African Union member states to allocating at least",
     "expected": "15% of their annual national budgets to the health sector. As of the 2020s, only a handful of African countries have met this target.",
     "rationale": "Landmark African health financing commitment — tests specific percentage and compliance knowledge."},

    {"id": "temp_12", "category": "Temporal Knowledge & Hallucination",
     "prompt": "Andela, the African tech talent company, was founded in",
     "expected": "2014 in Lagos, Nigeria by Jeremy Johnson, Christina Sass, Iyinoluwa Aboyeji, and Nadayar Enegesi. It trained and connected African software engineers with global companies.",
     "rationale": "African tech ecosystem knowledge — tests whether model knows founders and correct country."},

    {"id": "temp_13", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The meningitis belt in Africa refers to",
     "expected": "A region stretching from Senegal to Ethiopia across the sub-Saharan Sahel, where epidemic meningococcal meningitis occurs seasonally during the dry season (December-June). The introduction of MenAfriVac in 2010 dramatically reduced serogroup A outbreaks.",
     "rationale": "Region-specific epidemiological concept with vaccine history — tests multi-layered African health knowledge."},

    {"id": "temp_14", "category": "Temporal Knowledge & Hallucination",
     "prompt": "GeneXpert/Xpert MTB/RIF is a point-of-care diagnostic tool that detects",
     "expected": "Mycobacterium tuberculosis (TB) and rifampicin resistance simultaneously from a sputum sample in approximately 2 hours. It has been widely deployed across Africa since 2010 and significantly improved TB case detection.",
     "rationale": "Critical TB diagnostic technology deployed across Africa — tests specific medical device knowledge."},

    {"id": "temp_15", "category": "Temporal Knowledge & Hallucination",
     "prompt": "The African Development Bank (AfDB) is headquartered in",
     "expected": "Abidjan, Côte d'Ivoire (returned from temporary relocation in Tunis, Tunisia after the Ivorian civil war). It is the premier development finance institution for Africa, established in 1964.",
     "rationale": "Tests knowledge of African institutional geography including the relocation history."},

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 6: CODE GENERATION — DOMAIN-SPECIFIC (14 probes)
    # Hypothesis: Healthcare data standards and African tech patterns absent
    # ═══════════════════════════════════════════════════════════════════════

    {"id": "code_01", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Parse a FHIR R4 Patient resource into a flat dict\n# FHIR uses nested arrays for names, identifiers, telecom\ndef parse_fhir_patient(resource: dict) -> dict:\n    \"\"\"Extract name, DOB, gender, phone from FHIR Patient JSON.\"\"\"\n",
     "expected": "Navigate FHIR structure: resource['name'][0]['given'][0], resource['name'][0]['family'], resource['birthDate'], resource['gender'], resource['telecom'] filtered by system='phone'.",
     "rationale": "FHIR (HL7) is the global health interop standard but rare in code pretraining."},

    {"id": "code_02", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Calculate APGAR score for a newborn\n# 5 signs: Appearance, Pulse, Grimace, Activity, Respiration (each 0-2)\n# >=7 normal, 4-6 moderate, <4 critical\ndef apgar_score(appearance, pulse, grimace, activity, respiration):\n",
     "expected": "Validate inputs in {0,1,2}, sum to get total (0-10), return score with clinical interpretation.",
     "rationale": "Clinical + code intersection. APGAR is documented but cross-domain synthesis stresses 3B models."},

    {"id": "code_03", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Calculate pediatric drug dosage\n# Input: weight_kg, dose_per_kg_mg, frequency_per_day, duration_days\n# Return: single_dose_mg, daily_dose_mg, total_course_mg\ndef pediatric_dose(weight_kg, dose_per_kg_mg, frequency_per_day, duration_days):\n",
     "expected": "single = weight_kg * dose_per_kg_mg; daily = single * frequency_per_day; total = daily * duration_days. Should include validation (weight > 0, reasonable ranges).",
     "rationale": "Safety-critical medical calculation code. Tests whether model adds input validation and correct arithmetic."},

    {"id": "code_04", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Parse DHIS2 API response for aggregate malaria data\n# DHIS2 returns data in format: {\"rows\": [[orgUnit, period, value], ...]}\ndef parse_dhis2_malaria(response_json):\n    \"\"\"Extract org unit names and malaria case counts by period.\"\"\"\n",
     "expected": "Should iterate response_json['rows'], extract orgUnit (index 0), period (index 1), and value (index 2) into a structured format. May need to handle metadata for orgUnit name lookup.",
     "rationale": "DHIS2 API structure is unique to health informatics and almost certainly absent from code pretraining."},

    {"id": "code_05", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Implement offline-first data sync with conflict resolution\n# When device reconnects, sync local SQLite changes to remote PostgreSQL\n# Use last-write-wins for simple fields, merge for arrays\ndef sync_patient_records(local_db, remote_db, last_sync_timestamp):\n",
     "expected": "Should query local changes since last_sync, compare with remote versions, apply LWW conflict resolution using timestamps, handle merge for array fields, update sync timestamp.",
     "rationale": "Offline-first sync is core to health apps in low-connectivity settings — tests architectural pattern knowledge."},

    {"id": "code_06", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Calculate BMI and classify according to WHO categories\n# Underweight: <18.5, Normal: 18.5-24.9, Overweight: 25-29.9, Obese: >=30\ndef bmi_calculator(weight_kg: float, height_m: float) -> dict:\n",
     "expected": "Calculate BMI = weight / height². Return dict with bmi value and category. Should validate height > 0 and reasonable weight range.",
     "rationale": "Simple medical code but tests whether model gets the WHO classification boundaries exactly right."},

    {"id": "code_07", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Generate HL7 ADT^A01 admission message\n# ADT = Admit/Discharge/Transfer\ndef create_hl7_admit(patient_id, first_name, last_name, dob, gender, facility):\n    \"\"\"Return a valid HL7v2 ADT^A01 message string.\"\"\"\n",
     "expected": "Should construct pipe-delimited segments: MSH (message header with encoding chars), EVN (event type), PID (patient identification), PV1 (patient visit). HL7v2 uses | field separator and ^ component separator.",
     "rationale": "HL7v2 message construction is highly specific healthcare interoperability code — likely absent from pretraining."},

    {"id": "code_08", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: MUAC-based malnutrition screening classifier\n# MUAC < 115mm = SAM (red), 115-125mm = MAM (yellow), > 125mm = normal (green)\ndef classify_muac(muac_mm: float, age_months: int) -> dict:\n    \"\"\"Classify child (6-59 months) nutritional status from MUAC.\"\"\"\n",
     "expected": "Should validate age 6-59 months, apply MUAC thresholds, return status and color code. May include referral recommendation for SAM.",
     "rationale": "MUAC screening is daily community health work in Africa but the cutoffs are domain-specific."},

    {"id": "code_09", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Parse and validate a Ghana Health Service patient ID\n# Format: GHS-REGION-FACILITY-YEAR-SEQUENTIAL (e.g., GHS-AR-0042-2024-001234)\nimport re\ndef validate_ghs_id(patient_id: str) -> dict:\n",
     "expected": "Should use regex to parse the 5 components, validate region code exists, check year is reasonable, return parsed components or raise error.",
     "rationale": "Country-specific health ID format — completely fabricated but plausible. Tests whether model recognizes it can't know this and produces reasonable generic parsing vs hallucinating validation."},

    {"id": "code_10", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Convert ICD-10 code to human-readable diagnosis\n# Common tropical codes: B50 (P.falciparum malaria), A01 (typhoid), A09 (diarrhea)\ndef icd10_to_diagnosis(code: str) -> str:\n    \"\"\"Map ICD-10 code to diagnosis name. Return 'Unknown' if not found.\"\"\"\n",
     "expected": "Should contain a mapping dict with common ICD-10 codes including tropical diseases (B50-B54 for malaria species, A01 typhoid, A09 diarrhea, J18 pneumonia, etc). Should handle code not found.",
     "rationale": "ICD-10 mapping with emphasis on tropical disease codes — tests domain-specific knowledge in code form."},

    {"id": "code_11", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Estimate gestational age from last menstrual period (LMP)\nfrom datetime import date, timedelta\ndef gestational_age(lmp_date: date, current_date: date = None) -> dict:\n    \"\"\"Return weeks, days, trimester, and estimated due date.\"\"\"\n",
     "expected": "Calculate days since LMP, convert to weeks+days, determine trimester (T1: 0-12wk, T2: 13-27wk, T3: 28-40wk), compute EDD = LMP + 280 days (Naegele's rule).",
     "rationale": "Obstetric calculation code — tests whether model knows Naegele's rule and trimester boundaries."},

    {"id": "code_12", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: SMS-based health alert system for low-connectivity areas\n# Format: \"ALERT|facility_id|disease_code|case_count|date\"\ndef parse_sms_alert(sms_text: str) -> dict:\n    \"\"\"Parse structured SMS health alert. Validate fields.\"\"\"\n",
     "expected": "Split by pipe delimiter, validate 5 fields present, parse facility_id, validate disease_code format, convert case_count to int, parse date. Handle malformed messages gracefully.",
     "rationale": "SMS-based health reporting is still used across rural Africa — tests low-tech health informatics pattern."},

    {"id": "code_13", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: Calculate the Oxford severity score for community-acquired pneumonia (CRB-65)\n# C=Confusion, R=Respiratory rate>=30, B=BP systolic<90 or diastolic<=60, 65=age>=65\ndef crb65_score(confused: bool, resp_rate: int, systolic_bp: int, diastolic_bp: int, age: int) -> dict:\n",
     "expected": "Score 0-4: +1 if confused, +1 if resp_rate>=30, +1 if systolic<90 or diastolic<=60, +1 if age>=65. Return score and risk level (0=low, 1-2=moderate, 3-4=high/hospital admission).",
     "rationale": "Clinical scoring system implemented in code — tests whether model knows the exact CRB-65 criteria and thresholds."},

    {"id": "code_14", "category": "Code Generation (Domain-Specific)",
     "prompt": "# Python: CRDT-based counter for patient visit tracking in offline-first app\n# G-Counter: grow-only counter that merges by taking max per node\nclass GCounter:\n    def __init__(self, node_id: str):\n",
     "expected": "Should implement a G-Counter CRDT with: internal dict mapping node_id to count, increment() that increases own node's count, merge() that takes element-wise max across all nodes, value() that returns sum.",
     "rationale": "CRDTs are central to offline-first health app architecture but niche distributed systems concept."},
]

print(f"Total probes: {len(probes)}")
print(f"Categories: {len(set(p['category'] for p in probes))}")
from collections import Counter
for cat, n in Counter(p["category"] for p in probes).most_common():
    print(f"  {cat}: {n}")

Total probes: 104
Categories: 6
  Clinical & Biomedical Reasoning: 25
  Geographic & Cultural Bias: 20
  Multilingual Competence: 15
  Numerical & Logical Reasoning: 15
  Temporal Knowledge & Hallucination: 15
  Code Generation (Domain-Specific): 14


In [ ]:
results = []
for p in probes:
    out = complete(p["prompt"])
    print(f"\n[{p['id']}] {p['category']}")
    print(f"  PROMPT:   {p['prompt'][:100]}...")
    print(f"  MODEL:    {out[:200]}")
    print(f"  EXPECTED: {p['expected'][:200]}")
    results.append({
        "id": p["id"], "category": p["category"],
        "input": p["prompt"], "expected_output": p["expected"],
        "model_output": out, "blind_spot_rationale": p["rationale"],
        "error_type": "", "error_detail": ""
    })
print(f"\nDone — {len(results)} probes.")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



[geo_01] Geographic & Cultural Bias
  PROMPT:   The capital of Cameroon is Yaoundé. The largest city is Douala. The country has two official languag...
  MODEL:    French and English.

Cameroon is a landlocked country in Central Africa, bordered by Nigeria to the west; Chad to the northeast; the Central African Republic to the east; Equatorial Guinea to the nort
  EXPECTED: English and French. Cameroon is one of the few officially bilingual countries in Africa, a legacy of British and French colonial rule.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



[geo_02] Geographic & Cultural Bias
  PROMPT:   Ghana's national health insurance scheme (NHIS) was established in...
  MODEL:    1995 to provide universal access to healthcare services for all citizens. The NHIS is funded through a mandatory contribution from both employers and employees, as well as government subsidies.

The N
  EXPECTED: 2003 under the National Health Insurance Act (Act 650), providing financial access to healthcare funded through a levy, SSNIT contributions, and premiums.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...


[code_11] Code Generation (Domain-Specific)
  PROMPT:   # Python: Estimate gestational age from last menstrual period (LMP)
from datetime import date, timed...
  MODEL:       if not current_date:
        current_date = date.today()
    # Calculate the difference between dates in days
    days_diff = (current_date - lmp_date).days
    weeks_diff = int((days_diff / 7))  #
  EXPECTED: Calculate days since LMP, convert to weeks+days, determine trimester (T1: 0-12wk, T2: 13-27wk, T3: 28-40wk), compute EDD = LMP + 280 days (Naegele's rule).


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



[code_12] Code Generation (Domain-Specific)
  PROMPT:   # Python: SMS-based health alert system for low-connectivity areas
# Format: "ALERT|facility_id|dise...
  MODEL:       try:
        parts = sms_text.split('|')
        if len(parts) != 5 or not all(part.strip() for part in parts):
            raise ValueError("Invalid format")
        return {
            'type': '
  EXPECTED: Split by pipe delimiter, validate 5 fields present, parse facility_id, validate disease_code format, convert case_count to int, parse date. Handle malformed messages gracefully.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



[code_13] Code Generation (Domain-Specific)
  PROMPT:   # Python: Calculate the Oxford severity score for community-acquired pneumonia (CRB-65)
# C=Confusio...
  MODEL:       """
    Calculate the CRB-65 score.

    Args:
        confused (bool): Whether the patient is confused.
        resp_rate (int): The respiratory rate in breaths per minute.
        systolic_bp (in
  EXPECTED: Score 0-4: +1 if confused, +1 if resp_rate>=30, +1 if systolic<90 or diastolic<=60, +1 if age>=65. Return score and risk level (0=low, 1-2=moderate, 3-4=high/hospital admission).

[code_14] Code Generation (Domain-Specific)
  PROMPT:   # Python: CRDT-based counter for patient visit tracking in offline-first app
# G-Counter: grow-only ...
  MODEL:           """
        Initialize a new GCounter with the given node ID.
        """
        self._node_id = node_id  # Node identifier (e.g., IP address or hostname)
        self._counter = {}       # Di
  EXPECTED: Should implement a G-Counter CRDT with: internal 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: ERROR ANNOTATION
# ═══════════════════════════════════════════════════════════════════════════════
#
# HOW THIS WORKS:
# For each probe, you assign TWO things:
#   1. error_type  — what KIND of mistake (pick from the list below)
#   2. error_detail — one sentence explaining the SPECIFIC mistake
#
# ERROR TYPES:
#   "correct"            → Model got it right (or close enough)
#   "partial_correct"    → Some facts right, key details wrong or missing
#   "factual_error"      → States wrong facts (wrong dates, names, numbers)
#   "hallucination"      → Fabricates plausible-sounding but false info
#   "outdated_knowledge" → Info was once true but is now obsolete
#   "wrong_language"     → Responds in wrong language or code-switches
#   "arithmetic_error"   → Wrong numerical computation
#   "structural_error"   → Code ignores domain-specific structure
#   "format_deviation"   → Generates MCQ/exam instead of completing text
#   "repetition_loop"    → Gets stuck repeating
#   "vague_or_generic"   → Technically not wrong but lacks specificity
#
# I pre-filled these based on your actual outputs. Review each one —
# if you disagree, just change it. This is YOUR judgment call.
# ═══════════════════════════════════════════════════════════════════════════════

annotations = {
    # ─── GEOGRAPHIC & CULTURAL BIAS ─────────────────────────────────────
    "geo_01": ("correct", "Model correctly identified French and English as official languages."),
    "geo_02": ("factual_error", "Model said 1995; actual year is 2003 under Act 650. Funding mechanism also wrong."),
    "geo_03": ("hallucination", "Model said 1957 and described French-speaking majority rebelling against British rule — completely fabricated. Crisis began 2016-2017 as Anglophone protests."),
    "geo_04": ("factual_error", "Model said 2013; actual year is 2002. Did not mention founder Patrick Awuah or his Microsoft background."),
    "geo_05": ("partial_correct", "Correctly identified Ghanaian cedi and GHS but gave wrong abbreviation GHC and did not mention the 2007 redenomination."),
    "geo_06": ("factual_error", "Listed Somalia as observer instead of Burundi as full member, and omitted DRC which joined in 2022."),
    "geo_07": ("correct", "Correctly identified surface area and Akosombo Dam with reasonable date range."),
    "geo_08": ("format_deviation", "Generated MCQ options instead of completing the sentence. Did not state 2007 or Safaricom/Vodafone."),
    "geo_09": ("factual_error", "Model said January 2021; actual date is May 2019. Did not mention Accra secretariat."),
    "geo_10": ("correct", "Correctly identified Nollywood as second-largest after India."),
    "geo_11": ("partial_correct", "General direction correct but said 'Atlantic Ocean to Red Sea' instead of naming specific countries. Listed 8 countries which is reasonable."),
    "geo_12": ("vague_or_generic", "Gave generic description about community participation without mentioning CHOs, health compounds, or the Navrongo experiment."),
    "geo_13": ("hallucination", "Generated MCQ with French/English as options for 'indigenous' languages — confused official with indigenous. Actual answer: Fulfulde, Ewondo, Duala."),
    "geo_14": ("hallucination", "Fabricated that it was established by John Nkengasong (who is actually Africa CDC director, not connected to Navrongo). Did not mention the CHFP Project."),
    "geo_15": ("format_deviation", "Generated exam question format instead of completing text. Content direction was correct (Pan-Africanism, OAU)."),
    "geo_16": ("partial_correct", "Correctly identified northeastern Nigeria but did not mention specific states (Borno, Yobe, Adamawa) or cross-border impact on Cameroon, Chad, Niger."),
    "geo_17": ("vague_or_generic", "Mentioned English Language exams but did not name the 5 member countries or WASSCE specifically."),
    "geo_18": ("partial_correct", "Correctly placed Accra on Gulf of Guinea but understated population (said 2.5M vs 4M+ metro) and incorrectly described trade routes."),
    "geo_19": ("partial_correct", "Mentioned West African countries and France but did not distinguish the two CFA zones (XOF/XAF) or mention the euro peg."),
    "geo_20": ("partial_correct", "Named UCT and University of Ghana correctly but missed Ashesi, CMU Africa, KNUST, and the broader scope of the program."),

    # ─── CLINICAL & BIOMEDICAL REASONING ────────────────────────────────
    "med_01": ("format_deviation", "Generated MCQ with correct answer (malaria) as option A, but did not complete as text. Did not mention P. falciparum or RDT confirmation."),
    "med_02": ("correct", "Correctly identified ACT as first-line treatment."),
    "med_03": ("partial_correct", "Correctly identified pneumonia classification but then pivoted to a clinical vignette about chest X-ray contradicting IMCI. Did not state amoxicillin or referral criteria."),
    "med_04": ("format_deviation", "Generated MCQ format. Option A (eGFR <30) is correct but model did not commit to an answer in text completion."),
    "med_05": ("correct", "Correctly listed all 5 APGAR criteria and scoring interpretation (7-10 normal, <4 intervention)."),
    "med_06": ("vague_or_generic", "Mentioned salts, glucose, and water generically but did not specify potassium chloride, trisodium citrate, or the WHO osmolarity (245 mOsm/L)."),
    "med_07": ("partial_correct", "Mentioned hepatitis B, measles, polio but missed BCG, pentavalent, PCV, rotavirus, yellow fever — the distinctly African schedule components."),
    "med_08": ("correct", "Correctly stated 3-15 range and GCS <8 as poor prognostic indicator."),
    "med_09": ("format_deviation", "Generated MCQ with correct answer (Sub-Saharan Africa) as option B, but did not complete as text or mention the malaria-sickle cell evolutionary connection."),
    "med_10": ("partial_correct", "Reasonable definition but quoted WHO loosely. Did not specify midwife/doctor/nurse or mention ability to identify and refer complications."),
    "med_11": ("vague_or_generic", "Described IPTp concept generically but did not name sulfadoxine-pyrimethamine (SP), timing (second trimester), or WHO's 3-dose recommendation."),
    "med_12": ("partial_correct", "Got 3 of 4 delays approximately right but phrased differently and did not attribute to Thaddeus and Maine (1994). Said 'survival after delivery' instead of 'receiving adequate care at facility.'"),
    "med_13": ("correct", "Correctly described edema, hypoalbuminemia, and dermatosis for kwashiorkor; wasting without edema for marasmus."),
    "med_14": ("correct", "Correctly stated 140-159/90-99 mmHg for Stage 1 WHO classification."),
    "med_15": ("vague_or_generic", "Mentioned 'different mechanisms of action' but did not explain artemisinin's rapid action/short half-life vs partner drug's longer elimination. Incorrectly said partner drugs can be antibiotics/antihelminthics."),
    "med_16": ("partial_correct", "Correctly identified severe acute malnutrition but did not mention RUTF/Plumpy'Nut treatment or the 115-125mm MAM threshold."),
    "med_17": ("partial_correct", "Correctly identified PfHRP2 for P. falciparum but did not mention pLDH/aldolase for other species or the 15-20 min result time."),
    "med_18": ("correct", "Correctly identified onchocerciasis/river blindness and Onchocerca volvulus."),
    "med_19": ("partial_correct", "Listed cholera, measles, meningitis, malaria, TB, yellow fever — reasonable but included malaria/TB (which are weekly, not immediate) and missed Ebola, plague, AFP/polio."),
    "med_20": ("partial_correct", "Correctly listed peanut butter, sugar, vegetable oil, milk powder, micronutrients but did not specify caloric content (500kcal/92g) or no-refrigeration advantage."),
    "med_21": ("vague_or_generic", "Said 'tropical and subtropical regions' generically. Did not name specific West African countries (Ghana, Côte d'Ivoire) or WHO-recommended treatment (rifampicin + clarithromycin)."),
    "med_22": ("partial_correct", "Correctly described immediate ART upon diagnosis but did not mention the key distinction: regardless of CD4 count. Did not cite WHO 2015 adoption."),
    "med_23": ("partial_correct", "Correctly described monitoring labor progress but did not mention alert line/action line, cervical dilation plotting, or fetal descent tracking."),
    "med_24": ("vague_or_generic", "Mentioned monthly antimalarial doses during high transmission season and 70% reduction, but did not name the specific drugs (SP+AQ) or the July-October timing."),
    "med_25": ("correct", "Correctly described skin-to-skin contact, exclusive breastfeeding, and improved outcomes. Mentioned 'rooming in' which is related."),

    # ─── MULTILINGUAL COMPETENCE ────────────────────────────────────────
    "lang_01": ("format_deviation", "Entered <think> reasoning mode instead of producing the translation directly. May eventually reach correct answer but deviated from text completion."),
    "lang_02": ("format_deviation", "Generated MCQ with correct answer (A: child is ill, has fever) but did not complete as text."),
    "lang_03": ("format_deviation", "Entered <think> mode instead of translating directly. Correct approach detected but output is reasoning trace, not translation."),
    "lang_04": ("hallucination", "Translated as 'I am not you' — completely wrong. Actual meaning is 'I am in pain / my body hurts me.'"),
    "lang_05": ("format_deviation", "Entered <think> mode and dropped 'homa' from the breakdown. May not produce accurate translation of the medical terms."),
    "lang_06": ("format_deviation", "Generated MCQ options instead of interpreting the clinical complaint directly."),
    "lang_07": ("format_deviation", "Entered <think> mode. Correct direction but did not produce clean translation output."),
    "lang_08": ("partial_correct", "Translated as 'take me to hospital' instead of 'take him/her' — got the urgency right but confused the object pronoun 'am' (him/her) with first person."),
    "lang_09": ("format_deviation", "Entered <think> mode for Yoruba translation. Did not produce direct output."),
    "lang_10": ("hallucination", "Completely fabricated — translated a Wolof medical sentence as a proverb about 'the sun rises in the east.' Total failure on low-resource language."),
    "lang_11": ("format_deviation", "Entered <think> mode for English-to-French medical translation."),
    "lang_12": ("hallucination", "Translated 'Yaro yana da zazzabi' as 'He has a good reputation.' Actual meaning is 'The child has a fever.' Complete failure on Hausa medical vocabulary."),
    "lang_13": ("format_deviation", "Entered <think> mode instead of completing the Swahili medical phrase."),
    "lang_14": ("partial_correct", "Got 'child is crying' right but translated 'e no wan chop' as 'nothing to do' instead of 'doesn't want to eat.' Missed the eating/appetite component."),
    "lang_15": ("format_deviation", "Entered <think> mode for French public health statistics translation."),

    # ─── NUMERICAL & LOGICAL REASONING ──────────────────────────────────
    "num_01": ("correct", "Correctly computed 1,080 patients."),
    "num_02": ("format_deviation", "Entered <think> mode. Correct reasoning direction but did not produce final answer in text completion format."),
    "num_03": ("format_deviation", "Rephrased as a question instead of answering. Entered <think> mode."),
    "num_04": ("format_deviation", "Generated MCQ. Correct answer (112,500) was NOT among the options (listed 367,500 / 90,000 / 11,250 / 8,125) — all options are wrong."),
    "num_05": ("format_deviation", "Generated MCQ with options in grams (30g/60g/75g/90g). Correct answer is 3g — none of the options are correct. Entered <think> mode."),
    "num_06": ("format_deviation", "Generated MCQ. Correct answer (4,232) not among options (423,200 / 67,200 / 42,320 / 672,000) — all wrong by orders of magnitude."),
    "num_07": ("format_deviation", "Entered structured markdown mode with step-by-step breakdown instead of simple text completion."),
    "num_08": ("format_deviation", "Entered <think> mode. Correct approach described but no final answer in completion."),
    "num_09": ("format_deviation", "Generated MCQ. Option A (6 quarters) is correct. Entered <think> mode."),
    "num_10": ("format_deviation", "Generated MCQ. Correct answer (~109-110) not listed — closest option is D (112). Entered <think> mode."),
    "num_11": ("format_deviation", "Generated MCQ. All options (25/30/40/60/75) are wrong — correct answer is ~2.08 mL/min."),
    "num_12": ("vague_or_generic", "Set up the calculation framework but did not compute the actual answer (170,000). Added an unasked follow-up scenario."),
    "num_13": ("format_deviation", "Rephrased as question and entered <think> mode. Also incorrectly referenced 'iPhone' instead of generic smartphone."),
    "num_14": ("format_deviation", "Generated MCQ. Option A (24.93) is closest but incorrect — actual answer is 25.7 kg/m²."),
    "num_15": ("format_deviation", "Set up the formula in LaTeX but did not compute the actual percentage (56.7%)."),

    # ─── TEMPORAL KNOWLEDGE & HALLUCINATION ─────────────────────────────
    "temp_01": ("partial_correct", "Correctly described DHIS2 as health information management and open source, but did not mention HMIS specifically or its backbone role in African/South Asian health reporting."),
    "temp_02": ("correct", "Correctly stated 2015, 17 goals, 169 targets, 2030 agenda."),
    "temp_03": ("partial_correct", "Correctly described as open-source EMR for developing countries but said '10,000 sites' (plausible but unverified) and did not mention Partners In Health, Regenstrief Institute, or Kenya origin."),
    "temp_04": ("partial_correct", "Correct vision statement ('integrated, prosperous, peaceful Africa') but did not mention 2015 adoption date or specific flagship projects (AfCFTA, e-network, rail)."),
    "temp_05": ("correct", "Correctly stated 2021 and age range 5-17 months. Did not mention R21/Matrix-M (2023) but core fact is right."),
    "temp_06": ("correct", "Correctly identified Guinea, Liberia, Sierra Leone with accurate case/death figures (28,000/11,000)."),
    "temp_07": ("correct", "Correctly stated 2008, Kenya, post-election violence. Mentioned global use for disaster response and election monitoring."),
    "temp_08": ("correct", "Correctly stated March 11, 2020. Case count (4M by April 2020) is era-appropriate."),
    "temp_09": ("partial_correct", "Correct founding year (2002) but incorrectly described it as a partnership between WHO, UNICEF, and World Bank. The Global Fund is an independent financing mechanism, not a WHO/UNICEF partnership."),
    "temp_10": ("vague_or_generic", "Generated repetitive category headers (mHealth for Health Workers / Patients / Public Health) without describing the actual WHO classification framework."),
    "temp_11": ("factual_error", "Said 5% for education — completely wrong. Abuja Declaration committed 15% for HEALTH, not education. Wrong sector and wrong percentage."),
    "temp_12": ("factual_error", "Said 2015; actual founding was 2014. Did not name any founders (Jeremy Johnson, Iyinoluwa Aboyeji, etc.) or mention Lagos specifically."),
    "temp_13": ("partial_correct", "Correctly described Senegal-to-Ethiopia stretch and environmental/population risk factors, but did not mention dry season timing (Dec-June) or MenAfriVac introduction in 2010."),
    "temp_14": ("correct", "Correctly identified MTB and rifampicin resistance detection in 2 hours."),
    "temp_15": ("correct", "Correctly stated Abidjan, Côte d'Ivoire and 1964 founding. Did not mention the Tunis relocation but core facts are right."),

    # ─── CODE GENERATION (DOMAIN-SPECIFIC) ──────────────────────────────
    "code_01": ("partial_correct", "Attempted FHIR parsing with name/family extraction but treated 'family' as array (family[0]) when it is a string in FHIR R4. Did not extract telecom/phone filtering."),
    "code_02": ("partial_correct", "Correct sum logic but no input validation (0-2 range), no clinical interpretation (normal/moderate/critical), and added unrelated input() code."),
    "code_03": ("correct", "Correct arithmetic: single_dose = weight × dose, daily = frequency × single, total = days × daily. Used rounding. Lacked input validation but logic is sound."),
    "code_04": ("correct", "Correctly parsed rows with orgUnit/period/value extraction. Included row length validation."),
    "code_05": ("partial_correct", "Good structure with docstring and correct arg types (sqlite3/psycopg2), but output was truncated before showing actual sync logic or conflict resolution."),
    "code_06": ("partial_correct", "Started docstring correctly but output was truncated before showing BMI calculation or WHO category classification."),
    "code_07": ("structural_error", "Attempted HL7 message but crammed all segments into one MSH string with wrong field mapping. Did not produce separate EVN, PID, PV1 segments with proper delimiters."),
    "code_08": ("partial_correct", "Correct SAM/MAM/normal logic and color codes, but used <=114 instead of <115, and age validation started at 7 months instead of 6."),
    "code_09": ("correct", "Good regex with named groups, proper format validation. Pattern is reasonable for the described ID format."),
    "code_10": ("partial_correct", "Correct B50/A01/A09 mappings but included wrong Z32 code (pregnancy delivery — not an ICD-10 tropical disease). Output truncated before showing fallback logic."),
    "code_11": ("partial_correct", "Correct date difference and weeks calculation but output truncated before showing trimester classification or EDD (Naegele's rule)."),
    "code_12": ("correct", "Clean pipe-split with length validation, error handling via try/except, and structured return dict."),
    "code_13": ("partial_correct", "Good docstring and arg structure but output truncated before showing the actual scoring logic and risk classification."),
    "code_14": ("partial_correct", "Correct initialization with node_id and counter dict, but output truncated before showing increment(), merge(), and value() methods."),
}

# ═══════════════════════════════════════════════════════════════════════════════
# APPLY ANNOTATIONS TO RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

for r in results:
    if r["id"] in annotations:
        r["error_type"], r["error_detail"] = annotations[r["id"]]
    else:
        r["error_type"] = "unreviewed"
        r["error_detail"] = ""

# ═══════════════════════════════════════════════════════════════════════════════
# PRINT SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

from collections import Counter

print("=" * 60)
print("ERROR TYPE DISTRIBUTION")
print("=" * 60)
type_counts = Counter(r["error_type"] for r in results)
for et, n in type_counts.most_common():
    pct = n / len(results) * 100
    print(f"  {et:25s}  {n:3d}  ({pct:.1f}%)")
print(f"\n  TOTAL PROBES: {len(results)}")
print(f"  CORRECT:      {type_counts.get('correct', 0)} ({type_counts.get('correct', 0)/len(results)*100:.1f}%)")
print(f"  ANY ERROR:    {len(results) - type_counts.get('correct', 0)} ({(len(results) - type_counts.get('correct', 0))/len(results)*100:.1f}%)")

print("\n" + "=" * 60)
print("ERRORS BY CATEGORY")
print("=" * 60)
from itertools import groupby
for cat in sorted(set(r["category"] for r in results)):
    cat_results = [r for r in results if r["category"] == cat]
    correct = sum(1 for r in cat_results if r["error_type"] == "correct")
    print(f"\n  {cat}")
    print(f"    Correct: {correct}/{len(cat_results)} ({correct/len(cat_results)*100:.0f}%)")
    cat_errors = Counter(r["error_type"] for r in cat_results if r["error_type"] != "correct")
    for et, n in cat_errors.most_common():
        print(f"    {et}: {n}")

ERROR TYPE DISTRIBUTION
  partial_correct             32  (30.8%)
  format_deviation            28  (26.9%)
  correct                     22  (21.2%)
  vague_or_generic             9  (8.7%)
  factual_error                6  (5.8%)
  hallucination                6  (5.8%)
  structural_error             1  (1.0%)

  TOTAL PROBES: 104
  CORRECT:      22 (21.2%)
  ANY ERROR:    82 (78.8%)

ERRORS BY CATEGORY

  Clinical & Biomedical Reasoning
    Correct: 7/25 (28%)
    partial_correct: 10
    vague_or_generic: 5
    format_deviation: 3

  Code Generation (Domain-Specific)
    Correct: 4/14 (29%)
    partial_correct: 9
    structural_error: 1

  Geographic & Cultural Bias
    Correct: 3/20 (15%)
    partial_correct: 6
    factual_error: 4
    hallucination: 3
    format_deviation: 2
    vague_or_generic: 2

  Multilingual Competence
    Correct: 0/15 (0%)
    format_deviation: 10
    hallucination: 3
    partial_correct: 2

  Numerical & Logical Reasoning
    Correct: 1/15 (7%)
    format

In [ ]:
# # Export
with open("nanbeige4_blind_spots.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Exported {len(results)} records.")

Exported 104 records.


In [ ]:
# # Upload
from huggingface_hub import login, HfApi
login()
api = HfApi()


In [ ]:
REPO = "T-tchi0/nanbeige4-3b-base-blind-spot"

# Create repo
api.create_repo(repo_id=REPO, repo_type="dataset", exist_ok=True)

# Upload dataset
api.upload_file(
    path_or_fileobj="nanbeige4_blind_spots.jsonl",
    path_in_repo="data/nanbeige4_blind_spots.jsonl",
    repo_id=REPO,
    repo_type="dataset"
)
print("✓ Dataset uploaded")

# Read the content from the user-specified 'read me' file
with open("README.md", "r", encoding="utf-8") as f:
    readme_content = f.read()

# Write this content to README.md and upload it
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=REPO,
    repo_type="dataset"
)
print("✓ README uploaded from 'read me' file")

print(f"\n🔗 https://huggingface.co/datasets/{REPO}")

No files have been modified since last commit. Skipping to prevent empty commit.


✓ Dataset uploaded


/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:9897: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")
No files have been modified since last commit. Skipping to prevent empty commit.


✓ README uploaded from 'read me' file

🔗 https://huggingface.co/datasets/T-tchi0/nanbeige4-3b-base-blind-spot
